In [31]:
import os
import gc
import zarr
import yaml
import json
import numba
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm

from plotnine import *
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from anngeno import AnnGeno
# pl.Config.set_tbl_rows(15)

In [32]:
# Configuration and paths
or_threshold_pheno = 0.99
or_threshold_anno = 0.99

eur_samples_path = '/home/dnanexus/data_dir/unrelated_cauc_samples_3rd_degree.csv'

maf=1e-3
n = !wc -l $eur_samples_path
n_eur = int(n[0].split(' ')[0])
mac = maf*(2*n_eur)

config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"
with open(config_path) as f:
    config = yaml.safe_load(f)

# Read the variant consequence configuration
records = []
for group, consequences in config["variant_consequences"].items():
    for consequence in consequences:
        records.append({'variant_class': group, 'vep_consequence': consequence})
var_cons = pl.DataFrame(records)


# Use a list comprehension to flatten the nested dictionary into records
records = [
    {
        "category": category,
        "annotation": anno,
        "color": props["color"],
        "label": props["label"],
        "annotation_dir": props.get("direction", 1),
    }
    for category, annos in config["rare_variant_annotations"].items()
    for anno, props in annos.items()
]

# Create the DataFrame directly from the list of records
anno_config_df = pl.DataFrame(records)
all_annotation_list = anno_config_df.select(pl.col("annotation")).to_series().to_list()

anno_config_df

category,annotation,color,label,annotation_dir
str,str,str,str,i64
"""plof""","""loftee_hc""","""#1f77b4""","""LOFTEE HC""",1
"""missense""","""am_pathogenicity""","""#feb72d""","""AlphaMissense""",1
"""missense""","""esmscoremissense""","""#feb72d""","""ESM1v""",-1
"""genetic_diversity""","""cadd_raw""","""#1f77b4""","""CADD Raw""",1
"""conservation""","""gpn_star_llr_calibrated_mean""","""#942c80""","""GPN-Star""",-1
…,…,…,…,…
"""splicing""","""absplice_dna_max""","""#28a745""","""AbSplice (max)""",1
"""splicing""","""absplice2_max""","""#28a745""","""AbSplice2 (max)""",1
"""regulatory_nondir""","""promoterai_abs""","""#00A99D""","""PromoterAI abs""",1


In [ ]:
anno = pl.scan_parquet("/home/dnanexus/data_dir/genebass394genes_olink371genes_variants_union_annotated_250928.parquet")

# existing_annos = [c for c in all_annotation_list if c in anno.collect_schema().names()]

existing_annos = ['loftee_hc', 'am_pathogenicity']

# Filter variant classes
variant_class = "coding"
consequences_for_group = var_cons.filter(
    pl.col('variant_class') == variant_class
)['vep_consequence'].to_list()
filter_expression = pl.any_horizontal(
    (pl.col(c) == 1) for c in consequences_for_group if c in anno.collect_schema().names()
)

min_range = -2000
max_range = +0

anno = (
    anno
    .filter(
        # (
        #     (pl.col('consequence_missense_variant') == 1) |
        #     (pl.col('consequence_start_lost') == 1) |
        #     (pl.col('consequence_stop_gained') == 1) |
        #     (pl.col('consequence_stop_lost') == 1) |
        #     (pl.col('consequence_splice_donor_variant') == 1) |
        #     (pl.col('consequence_splice_acceptor_variant') == 1)
        # ) &
        
        # Only SNPs
        (pl.col('ref').str.len_chars()==1) & (pl.col('alt').str.len_chars()==1)
    )
    .select(
        set(['id', 'region', 'TSS', 'Strand', 'gene_length', 'gene_name', 'dist_to_tss']).union(set(existing_annos))
    )
    .collect(engine='streaming')
    # .drop_nulls()
)

anno

gene_name,id,am_pathogenicity,loftee_hc,gene_length,Strand,dist_to_tss,region,TSS
str,str,f32,i8,i64,cat,i64,str,i64
"""LRP2""","""chr2:169197167:T:G""",0.0,0,235427,"""-""",165367,"""ENSG00000081479""",169362534
"""LRP2""","""chr2:169204292:A:T""",0.0,0,235427,"""-""",158242,"""ENSG00000081479""",169362534
"""ABCB11""","""chr2:169030531:C:T""",0.0,0,115828,"""-""",793,"""ENSG00000073734""",169031324
"""ABCB11""","""chr2:168910825:T:G""",0.0,0,115828,"""-""",120499,"""ENSG00000073734""",169031324
"""ABCB11""","""chr2:168969544:T:C""",0.0607,0,115828,"""-""",61780,"""ENSG00000073734""",169031324
…,…,…,…,…,…,…,…,…
"""GCKR""","""chr2:27526161:C:A""",0.0,0,26847,"""+""",29323,"""ENSG00000084734""",27496838
"""GCKR""","""chr2:27524879:T:C""",0.0,0,26847,"""+""",28041,"""ENSG00000084734""",27496838
"""TRIM40""","""chr6:30150733:A:C""",0.0,0,12613,"""+""",14610,"""ENSG00000204614""",30136123


In [34]:
melted_anno = (
    anno
    .with_columns(
        am_pathogenicity = pl.col('am_pathogenicity') >= 0.8
    )

    .unpivot(
        index=["id", "region"],
        on=existing_annos,
        variable_name="annotation",
        value_name="annotation_score"
    )
    .with_columns(
        pl.col("annotation_score").cast(pl.Float32),
        pl.col("region").cast(pl.Utf8),
    )

    # Merge with annotation configuration to get direction and filter
    .join(
        anno_config_df,
        on="annotation",
        how="left"
    )
    .with_columns(
        annotation_score_dircor = pl.col('annotation_score') * pl.col("annotation_dir")
    )
)

melted_anno

id,region,annotation,annotation_score,category,color,label,annotation_dir,annotation_score_dircor
str,str,str,f32,str,str,str,i64,f64
"""chr2:169197167:T:G""","""ENSG00000081479""","""loftee_hc""",0.0,"""plof""","""#1f77b4""","""LOFTEE HC""",1,0.0
"""chr2:169204292:A:T""","""ENSG00000081479""","""loftee_hc""",0.0,"""plof""","""#1f77b4""","""LOFTEE HC""",1,0.0
"""chr2:169030531:C:T""","""ENSG00000073734""","""loftee_hc""",0.0,"""plof""","""#1f77b4""","""LOFTEE HC""",1,0.0
"""chr2:168910825:T:G""","""ENSG00000073734""","""loftee_hc""",0.0,"""plof""","""#1f77b4""","""LOFTEE HC""",1,0.0
"""chr2:168969544:T:C""","""ENSG00000073734""","""loftee_hc""",0.0,"""plof""","""#1f77b4""","""LOFTEE HC""",1,0.0
…,…,…,…,…,…,…,…,…
"""chr2:27526161:C:A""","""ENSG00000084734""","""am_pathogenicity""",0.0,"""missense""","""#feb72d""","""AlphaMissense""",1,0.0
"""chr2:27524879:T:C""","""ENSG00000084734""","""am_pathogenicity""",0.0,"""missense""","""#feb72d""","""AlphaMissense""",1,0.0
"""chr6:30150733:A:C""","""ENSG00000204614""","""am_pathogenicity""",0.0,"""missense""","""#feb72d""","""AlphaMissense""",1,0.0


In [35]:
melted_anno.select(['id', 'annotation', 'annotation_score']).unique().filter(pl.col('annotation') == 'loftee_hc').select('annotation_score').sum()

annotation_score
f32
21519.0


In [36]:
melted_anno.select(['id', 'annotation', 'annotation_score']).unique().filter(pl.col('annotation') == 'am_pathogenicity').select('annotation_score').sum()

annotation_score
f32
56609.0


In [37]:
appv = pl.scan_parquet("/home/dnanexus/data_dir/appv_files/avg_pheno_per_var_quantitative_EUR_genebass1e6_PRScorr_with_percentiles.parquet")

# Create a lazy frame with the unique keys
anno_keys = anno.select(pl.col('id').unique()).lazy()

# Chain the filter and the much faster semi join
appv = (
    appv
        .join(
        anno_keys, on='id', how='semi'
    )
    .filter(
        pl.col('n_individuals') <= mac
    )
    .select(
        ['id', 'phenotype', 'mean_pheno_value_ptile', 'n_individuals']
    )
)

# unique_phenotypes = appv.select('phenotype').unique().collect(engine='streaming').to_series()

In [38]:
# Get gene trait associations

plof = pl.read_parquet('/home/dnanexus/data_dir/association_files/rvat_EUR_500k_regenie.parquet').with_columns(
    phenotype = (pl.col('trait') + '_int'),
    region = pl.col('gene_id'),
    rvat_pval = (10** -pl.col("neg_log10p")),
).filter(
    (pl.col('trait_type') == 'quantitative')
)

loftee_corr = pl.read_parquet("/home/dnanexus/data_dir/association_files/loftee_correlation_quantitative_wgs_EUR_genebass1e6_maf1e3_snp_consistent.parquet")

gene_trait_df = loftee_corr.join(plof[['region', 'phenotype', 'beta', 'rvat_pval']], on=['region', 'phenotype'], how='inner').filter(
    pl.col('loftee_corr')*pl.col('beta') > 0
).with_columns(
    corr_dir = pl.col('loftee_corr')/pl.col('loftee_corr').abs()
)

# gene_trait_df = gene_trait_df.head()
gene_trait_df

region,gene_name,phenotype,loftee_corr,n_variants,beta,rvat_pval,corr_dir
str,str,str,f64,u64,f64,f64,f64
"""ENSG00000116183""","""PAPPA2""","""arm_fatfree_mass_right_int""",-0.014747,82603,-0.195409,9.1637e-8,-1.0
"""ENSG00000129083""","""COPB1""","""arm_fatfree_mass_right_int""",-0.014338,13594,-0.397279,0.006626,-1.0
"""ENSG00000100578""","""KIAA0586""","""arm_fatfree_mass_right_int""",-0.010787,28541,-0.058463,0.000305,-1.0
"""ENSG00000157766""","""ACAN""","""arm_fatfree_mass_right_int""",-0.036259,20244,-0.436618,6.0395e-11,-1.0
"""ENSG00000140443""","""IGF1R""","""arm_fatfree_mass_right_int""",-0.013644,82469,-0.412619,2.8609e-8,-1.0
…,…,…,…,…,…,…,…
"""ENSG00000112077""","""RHAG""","""reticulocyte_count_int""",0.033239,9849,0.952411,1.8038e-41,1.0
"""ENSG00000029534""","""ANK1""","""reticulocyte_count_int""",0.02685,54620,0.656849,2.1682e-10,1.0
"""ENSG00000197969""","""VPS13A""","""reticulocyte_count_int""",0.003971,55477,0.176348,2.1188e-7,1.0


In [39]:
gene_trait_df['region'].n_unique(), gene_trait_df.shape[0]

(352, 1176)

In [40]:
# --- 1. Lazily prepare the filter keys ---
region_keys = gene_trait_df.lazy().select(pl.col('region').unique())
anno_ids_lazy = melted_anno.lazy().join(
    region_keys, on='region', how='semi'
).select(pl.col('id').unique())


# --- Build main query (same as before, but stop before group_by) ---
gp_lazy = (
    appv
    .join(anno_ids_lazy, on="id", how="semi")
    .join(
        melted_anno.lazy().drop([c for c in melted_anno.columns if 'is_nan' in c]), 
        on="id", 
        how="inner"
    )
    .join(
        gene_trait_df.lazy(), 
        on=["region", "phenotype"], 
        how="inner"
    )
    .with_columns(
        mean_pheno_value_dircor_ptile = pl.col('mean_pheno_value_ptile') * pl.col('loftee_corr')
    )
    
    .group_by(["annotation", "region", "gene_name", "phenotype"])
    .agg(
        n_dis_above_cutoff = ((pl.col("annotation_score_dircor")==1) & (pl.col("mean_pheno_value_dircor_ptile") >= or_threshold_pheno)).sum(),
        n_notdis_above_cutoff = ((pl.col("annotation_score_dircor")==1) & (pl.col("mean_pheno_value_dircor_ptile") < or_threshold_pheno)).sum()
    )
    .with_columns(
        odds_ratio = ((pl.col("n_dis_above_cutoff") + 1) / (pl.col("n_notdis_above_cutoff")+1)) / (1 - or_threshold_pheno)
    )
)

# Execute
print("Executing with odds ratios...")
or_df = gp_lazy.collect(engine='streaming')
or_df

Executing with odds ratios...


annotation,region,gene_name,phenotype,n_dis_above_cutoff,n_notdis_above_cutoff,odds_ratio
str,str,str,str,u64,u64,f64
"""loftee_hc""","""ENSG00000127585""","""FBXL16""","""arm_fat_percentage_right_int""",0,4,20.0
"""am_pathogenicity""","""ENSG00000187045""","""TMPRSS6""","""haemoglobin_concentration_int""",0,0,100.0
"""am_pathogenicity""","""ENSG00000087237""","""CETP""","""apolipoprotein_a_int""",0,7,12.5
"""am_pathogenicity""","""ENSG00000065600""","""PACC1""","""sitting_height_int""",0,20,4.761905
"""am_pathogenicity""","""ENSG00000136859""","""ANGPTL2""","""standing_height_int""",0,68,1.449275
…,…,…,…,…,…,…
"""am_pathogenicity""","""ENSG00000145850""","""TIMD4""","""cholesterol_int""",0,17,5.555556
"""am_pathogenicity""","""ENSG00000236320""","""SLFN14""","""mean_platelet_thrombocyte_volu…",0,15,6.25
"""am_pathogenicity""","""ENSG00000118777""","""ABCG2""","""urate_int""",0,65,1.515152


In [41]:
plt_df = (
    or_df
    .drop_nans()
    .with_columns(
        n_gene_phenos = pl.len().over('annotation'),
        avg_odds_ratio = pl.col('odds_ratio').mean().over('annotation'),
        std_odds_ratio = pl.col('odds_ratio').std().over('annotation'),
    )
    .with_columns(
        std_ci_upper = pl.col('avg_odds_ratio') + 1.96 * (pl.col('std_odds_ratio')),
        std_ci_lower = pl.col('avg_odds_ratio') - 1.96 * (pl.col('std_odds_ratio')),
    )
    .select(['annotation', 'n_gene_phenos', 'avg_odds_ratio', 'std_odds_ratio', 'std_ci_upper', 'std_ci_lower'])
    .unique()
)
plt_df

annotation,n_gene_phenos,avg_odds_ratio,std_odds_ratio,std_ci_upper,std_ci_lower
str,u64,f64,f64,f64,f64
"""am_pathogenicity""",1176,8.966652,23.514658,55.055382,-37.122078
"""loftee_hc""",1176,6.735885,7.426953,21.292713,-7.820942


In [13]:
plt_df = (
    or_df
    .drop_nans()
    .with_columns(
        n_gene_phenos = pl.len().over('annotation'),
        avg_odds_ratio = pl.col('odds_ratio').mean().over('annotation'),
        std_odds_ratio = pl.col('odds_ratio').std().over('annotation'),
    )
    .with_columns(
        std_ci_upper = pl.col('avg_odds_ratio') + 1.96 * (pl.col('std_odds_ratio')),
        std_ci_lower = pl.col('avg_odds_ratio') - 1.96 * (pl.col('std_odds_ratio')),
    )
    .select(['annotation', 'n_gene_phenos', 'avg_odds_ratio', 'std_odds_ratio', 'std_ci_upper', 'std_ci_lower'])
    .unique()
)
plt_df

annotation,n_gene_phenos,avg_odds_ratio,std_odds_ratio,std_ci_upper,std_ci_lower
str,u64,f64,f64,f64,f64
"""loftee_hc""",1176,6.735885,7.426953,21.292713,-7.820942
"""am_pathogenicity""",1176,8.966652,23.514658,55.055382,-37.122078
